# Example queries: `building_kws` (resstock_oedi)

Auto-generated from `tests/query_snapshots/building_kws.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [ ]:
from pathlib import Path
from buildstock_query import BuildStockQuery
import pandas as pd


## Construct the BuildStockQuery object

`cache_folder` points at the snapshot test cache directory so this
notebook reuses parquets that the test suite has already downloaded
from Athena. Queries that are already cached return immediately;
anything new still hits Athena.


In [ ]:
# This notebook lives in `tests/example_notebooks/`; the snapshot test
# cache is its sibling `tests/query_snapshots/resstock_oedi_cache/`. Resolve
# the path relative to the notebook directory (`_dh[0]` is set by
# IPython at kernel startup; falls back to CWD outside Jupyter).
_NB_DIR = Path(_dh[0] if "_dh" in globals() else ".").resolve()
_CACHE = (_NB_DIR / "../query_snapshots/resstock_oedi_cache").resolve()
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "resstock_2024_amy2018_release_2",
    buildstock_type="resstock",
    db_schema="resstock_oedi_vu",
    skip_reports=True,
    cache_folder=str(_CACHE),
)


## `building_kws_noon_grid_aligned_co`

agg.get_building_average_kws_at at noon (12.0) for three days, restricted to CO. Hour aligns to the 15-min grid (12*3600 % 900 == 0) so the method returns a single-element SQL list — pins the 'exact_times' branch. The restrict + upgrade=0 filter on the TS join (added 2026-04-25) caps Athena scan to a single state's partition; without those, this query was a 3.2 TB landmine.


In [ ]:
result = bsq.agg.get_building_average_kws_at(
    at_hour=12.0,
    at_days=[1, 100, 200],
    enduses=['out.electricity.total.energy_consumption'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


# shape: (9425, 4)
 bldg_id  sample_count  units_count  electricity.total.energy_consumption
      13             3   756.904916                            156.763418
      16             3   756.904916                            130.524048
      33             3   756.904916                            486.773962
     128             3   756.904916                            239.854758
     182             3   756.904916                            110.003514


## `building_kws_off_grid_interpolated_co`

agg.get_building_average_kws_at at 12.1h (off the 15-min grid) for three days, CO restrict. Returns two SQL strings (lower + upper timestamps for interpolation), joined with MULTI_SQL_SEPARATOR. Pins the two-query interpolation branch.


In [ ]:
result = bsq.agg.get_building_average_kws_at(
    at_hour=12.1,
    at_days=[1, 100, 200],
    enduses=['out.electricity.total.energy_consumption'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


# shape: (9425, 4)
 bldg_id  sample_count  units_count  electricity.total.energy_consumption
      13             3   756.904916                            156.628857
      16             3   756.904916                            138.732261
      33             3   756.904916                            475.605409
     128             3   756.904916                            231.915666
     182             3   756.904916                            105.832127
